In [2]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
print(next(iter(dataset)))

{'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1506x1102 at 0xE80455319150>], 'texts': [{'user': 'What is the primary key of the Equipment entity?', 'assistant': 'The primary key of the Equipment entity is EquipmentID, which uniquely identifies each piece of equipment in the diagram.\nAnswer: EquipmentID'}, {'user': 'What is the relationship between Equipment and Maintenance Record?', 'assistant': 'The relationship between Equipment and Maintenance Record is labeled as "Maintained By", indicating that one piece of equipment can have multiple maintenance records associated with it.\nAnswer: Maintained By'}, {'user': 'How many relationships does the Supplier entity have?', 'assistant': 'The Supplier entity has two relationships: "Supplied By" with Equipment and "Delivered By" with Logistics Company, indicating it serves as a midpoint in both flows.\nAnswer: 2'}, {'user': 'What is the cardinality of the relationship between Equipment and Supplier?', 'assistant': 'The car

In [3]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoProcessor, AutoTokenizer, AutoModelForCausalLM

class VisionLanguageModel(nn.Module):
    def __init__(self, vision_ckpt, language_ckpt, modality_input_dim=768, modality_output_dim=576):
        super().__init__()
        # only take vision backbone
        self.vision_encoder = AutoModel.from_pretrained(vision_ckpt).vision_model
        self.vision_processor = AutoProcessor.from_pretrained(vision_ckpt)
        self.modality_projector = nn.Linear(modality_input_dim, modality_output_dim, bias=False)
        self.tokenizer = AutoTokenizer.from_pretrianed(language_ckpt)
        self.llm = AutoModelForCausalLM.from_pretrained(language_ckpt)

    def forward(self, text, image):
        processed_img = self.vision_processor(images=[image], return_tensors="pt").to(self.llm.device)
        image_embd = self.vision_encoder(**processed_img).last_hidden_state
        # match llm dtype
        image_embd = self.modality_projector(image_embd).to(dtype=self.llm.dtype)
        imput_ids = self.tokenizer(text, return_tensors="pt").input_ids.to(self.llm.device)

        token_embd = self.llm.model.embed_tokens(input_ids)
        combined_embd = torch.cat((image_embd, token_embd), dim=1)
        # Concatenate image embeddings to token embeddings

        logits = self.llm(inputs_embeds=combined_embd).logits
        shift_logits = logits[:, image_embd.size(1):-1, :].contigous()
        # remove images
        shift_labels = input_ids[:, 1:].contigous()

        loss = nn.functional.cross_entropy(shift_logits.reshape(-1,
                                   shift_logits.size(-1)),
                                   shift_labels.reshape(-1))

        return logits, loss